In [1]:
import pandas as pd

sales = pd.read_csv("sales.csv", parse_dates=["Date"])
sales = sales.sort_values("Date").reset_index(drop=True)

# Check completeness
full_range = pd.date_range(sales["Date"].min(), sales["Date"].max(), freq="D")
missing = full_range.difference(sales["Date"])
print(f"Missing dates: {len(missing)}")  # should be 0

# Derive gross margin ratio (safe to use as train feature)
sales["gross_margin"] = (sales["Revenue"] - sales["COGS"]) / sales["Revenue"]

master = sales.copy()


Missing dates: 0


In [2]:
import numpy as np
import holidays

d = master["Date"]
master["dow"] = d.dt.dayofweek  # 0=Mon
master["month"] = d.dt.month
master["quarter"] = d.dt.quarter
master["year"] = d.dt.year
master["week"] = d.dt.isocalendar().week.astype(int)
master["is_weekend"] = (master["dow"] >= 5).astype(int)
master["days_to_me"] = d.dt.days_in_month - d.dt.day
master["is_month_start"] = d.dt.is_month_start.astype(int)
master["is_payday"] = d.dt.day.isin([1, 15]).astype(int)

# Cyclical encoding (no ordinal assumption)
master["sin_month"] = np.sin(2 * np.pi * master["month"] / 12)
master["cos_month"] = np.cos(2 * np.pi * master["month"] / 12)
master["sin_dow"] = np.sin(2 * np.pi * master["dow"] / 7)
master["cos_dow"] = np.cos(2 * np.pi * master["dow"] / 7)

# Vietnamese holidays
vn_holidays = set(holidays.VN(years=range(2012, 2025)).keys())

master["Date"] = pd.to_datetime(master["Date"])
# encode 0/1
master["is_holiday"] = master["Date"].dt.date.isin(vn_holidays).astype(int)

In [3]:
rev = master["Revenue"]  # must be sorted by Date already

# Point-in-time lags
for lag in [1, 7, 14, 30, 365]:
    master[f"lag_{lag}"] = rev.shift(lag)

# Same-weekday lag (captures weekly rhythm)
master["lag_7d_weekday"] = rev.shift(7)

# Rolling statistics (shift FIRST, then roll)
shifted = rev.shift(1)
for w in [7, 14, 30]:
    master[f"roll_mean_{w}"] = shifted.rolling(w, min_periods=1).mean()
for w in [7, 30]:
    master[f"roll_std_{w}"] = shifted.rolling(w, min_periods=1).std()

# Exponential weighted mean (recent days weighted more)
master["ewm_03"] = shifted.ewm(alpha=0.3, adjust=False).mean()
master["ewm_01"] = shifted.ewm(alpha=0.1, adjust=False).mean()

# YoY: same day last year (handles annual seasonality directly)
master["lag_365"] = rev.shift(365)

# Revenue momentum (rate of change over last 7 days)
master["momentum_7"] = rev.shift(1) / rev.shift(8) - 1


In [4]:
traffic = pd.read_csv("web_traffic.csv", parse_dates=["date"])

# Aggregate totals per day (across all sources)
daily_agg = (
    traffic.groupby("date")
    .agg(
        total_sessions=("sessions", "sum"),
        total_visitors=("unique_visitors", "sum"),
        total_pageviews=("page_views", "sum"),
        avg_bounce_rate=("bounce_rate", "mean"),
        avg_duration_sec=("avg_session_duration_sec", "mean"),
    )
    .reset_index()
)

# Pivot sessions per source → wide format
sessions_wide = traffic.pivot_table(
    index="date",
    columns="traffic_source",
    values="sessions",
    aggfunc="sum",
    fill_value=0,
).reset_index()
sessions_wide.columns = ["date"] + [
    f"sess_{c.replace(' ', '_')}" for c in sessions_wide.columns[1:]
]

# Merge both and compute share features
traffic_feat = daily_agg.merge(sessions_wide, on="date", how="left")
traffic_feat["paid_share"] = traffic_feat.get("sess_paid_search", 0) / traffic_feat[
    "total_sessions"
].replace(0, 1)

# Join to master
master = master.merge(
    traffic_feat.rename(columns={"date": "Date"}), on="Date", how="left"
)

In [5]:
promos = pd.read_csv("promotions.csv", parse_dates=["start_date", "end_date"])

# Explode each promo into one row per day
rows = []
for _, row in promos.iterrows():
    dates = pd.date_range(row["start_date"], row["end_date"], freq="D")
    tmp = pd.DataFrame({"Date": dates})
    tmp["discount_value"] = row["discount_value"]
    tmp["promo_type"] = row["promo_type"]
    tmp["stackable_flag"] = row["stackable_flag"]
    rows.append(tmp)

daily_promos = pd.concat(rows, ignore_index=True)

# Aggregate per day
promo_feat = (
    daily_promos.groupby("Date")
    .agg(
        promo_active=("discount_value", "count"),  # >0 = active
        n_active_promos=("discount_value", "count"),
        max_discount=("discount_value", "max"),
        has_stackable=("stackable_flag", "max"),
    )
    .reset_index()
)
promo_feat["promo_active"] = (promo_feat["promo_active"] > 0).astype(int)

# Days since last promo ended / days to next promo starts
# (useful leading/lagging indicators)
all_promo_ends = pd.DatetimeIndex(promos["end_date"].unique())
master["days_since_promo"] = master["Date"].apply(
    lambda d: (
        (d - all_promo_ends[all_promo_ends <= d].max()).days
        if any(all_promo_ends <= d)
        else -1
    )
)

master = master.merge(promo_feat, on="Date", how="left")
master[["promo_active", "n_active_promos", "max_discount", "has_stackable"]] = master[
    ["promo_active", "n_active_promos", "max_discount", "has_stackable"]
].fillna(0)


In [6]:
inv = pd.read_csv("inventory.csv", parse_dates=["snapshot_date"])

# Aggregate to one row per snapshot_date (across all products)
inv_daily = (
    inv.groupby("snapshot_date")
    .agg(
        avg_fill_rate=("fill_rate", "mean"),
        pct_stockout=("stockout_flag", "mean"),
        avg_stockout_days=("stockout_days", "mean"),
        avg_sell_through=("sell_through_rate", "mean"),
        pct_overstock=("overstock_flag", "mean"),
    )
    .reset_index()
    .rename(columns={"snapshot_date": "Date"})
)

# Reindex to daily frequency and forward-fill
inv_daily = inv_daily.set_index("Date")
inv_daily = (
    inv_daily.reindex(
        pd.date_range(inv_daily.index.min(), inv_daily.index.max(), freq="D")
    )
    .ffill()
    .reset_index()
    .rename(columns={"index": "Date"})
)

# Join
master = master.merge(inv_daily, on="Date", how="left")

# Lag inventory features by 1 month (inventory state this month
# was decided before sales happen — use previous month's snapshot)
for col in ["avg_fill_rate", "pct_stockout", "avg_stockout_days"]:
    master[f"{col}_lag1m"] = master[col].shift(30)


In [7]:
# --- DROP lag warmup rows (first 365 rows have NaN lags) ---
master = master.dropna(subset=["lag_7", "lag_30"]).reset_index(drop=True)

# --- LEAKAGE CHECK ---
# Rule: no column derived from future Revenue/COGS values.
# Same-day Revenue and COGS must NOT be features (they're targets/test labels).
forbidden = ["Revenue", "COGS", "gross_margin"]
feature_cols = [c for c in master.columns if c not in forbidden + ["Date"]]

# Quick check: correlation of each feature with Revenue
# Any feature with corr > 0.99 is suspicious (likely leakage)
corr = master[feature_cols].corrwith(master["Revenue"])
print(corr.sort_values(ascending=False).head(10))

# --- TIME-BASED TRAIN / VALIDATION SPLIT ---
# Never use random split on time series — future leaks into past
val_cutoff = pd.Timestamp("2022-07-01")  # last 6 months as val
train = master[master["Date"] < val_cutoff]
val = master[master["Date"] >= val_cutoff]

X_train = train[feature_cols]
y_train = train["Revenue"]
X_val = val[feature_cols]
y_val = val["Revenue"]

# --- SAVE ---
master.to_csv("master_train.csv", index=False)
print(f"Master shape: {master.shape}")
print(f"Train: {len(train)} rows | Val: {len(val)} rows")
print(f"Features: {len(feature_cols)}")
train.to_csv("Train_test.csv")
val.to_csv("Val_test.csv")

lag_1           0.865692
lag_365         0.789784
ewm_03          0.784604
ewm_01          0.718346
roll_mean_7     0.695612
roll_mean_30    0.683310
roll_mean_14    0.670527
lag_30          0.652300
roll_std_30     0.616257
lag_14          0.496840
dtype: float64
Master shape: (3803, 57)
Train: 3619 rows | Val: 184 rows
Features: 53


In [8]:
master["is_holiday"].value_counts()

is_holiday
0    3663
1     140
Name: count, dtype: int64

In [9]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 3803 entries, 0 to 3802
Data columns (total 57 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Date                     3803 non-null   datetime64[us]
 1   Revenue                  3803 non-null   float64       
 2   COGS                     3803 non-null   float64       
 3   gross_margin             3803 non-null   float64       
 4   dow                      3803 non-null   int32         
 5   month                    3803 non-null   int32         
 6   quarter                  3803 non-null   int32         
 7   year                     3803 non-null   int32         
 8   week                     3803 non-null   int64         
 9   is_weekend               3803 non-null   int64         
 10  days_to_me               3803 non-null   int32         
 11  is_month_start           3803 non-null   int64         
 12  is_payday                3803 non-null   int6